In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/hr_attrition_v1.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,promotion_gap_ratio,manager_tenure_ratio,distance_per_income,overtime_flag,travel_frequently_flag,single_flag,male_flag,age_group,income_group,tenure_group
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,0.000000,0.714286,0.000167,1,0,1,0,36-45,중상,6-10년
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,0.090909,0.636364,0.001559,0,1,0,1,46-55,중상,6-10년
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,0.000000,0.000000,0.000956,1,0,1,1,36-45,저소득,0-2년
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,0.333333,0.000000,0.001031,1,1,0,0,26-35,저소득,6-10년
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,0.666667,0.666667,0.000577,0,0,0,1,26-35,중하,0-2년


In [2]:
drop_cols_v2 = [
    "Attrition",
    "attrition_flag",
    "overtime_flag",
    "travel_frequently_flag",
    "single_flag",
    "male_flag",
    "DailyRate",
    "MonthlyRate",
    "HourlyRate"
]

X_v2 = df.drop(columns=drop_cols_v2)
y = df["attrition_flag"]

print(X_v2.shape, y.shape)
print(X_v2.columns.tolist())

(1470, 35) (1470,)
['Age', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'income_log', 'tenure_ratio', 'promotion_gap_ratio', 'manager_tenure_ratio', 'distance_per_income', 'age_group', 'income_group', 'tenure_group']


In [3]:
numeric_cols = X_v2.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_v2.select_dtypes(include=["object"]).columns.tolist()

print("숫자형 변수 수:", len(numeric_cols))
print(numeric_cols)
print()
print("범주형 변수 수:", len(categorical_cols))
print(categorical_cols)

숫자형 변수 수: 25
['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'income_log', 'tenure_ratio', 'promotion_gap_ratio', 'manager_tenure_ratio', 'distance_per_income']

범주형 변수 수: 10
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'age_group', 'income_group', 'tenure_group']


C:\Users\hissc\AppData\Local\Temp\ipykernel_6032\4267104118.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_v2.select_dtypes(include=["object"]).columns.tolist()


In [4]:
df_v2 = pd.concat([X_v2, y], axis=1)

output_path = "../data/processed/hr_attrition_v2.csv"
df_v2.to_csv(output_path, index=False, encoding="utf-8-sig")

print(output_path)
print(df_v2.shape)

../data/processed/hr_attrition_v2.csv
(1470, 36)


v2 변수셋 정리 메모

- LightGBM baseline 결과, 일부 rate 변수(DailyRate, MonthlyRate, HourlyRate)가 중요도 상위에 나타났으나 해석력 제한적
- 원본 범주형 변수와 의미가 중복되는 일부 이진 파생변수 제거
- v2 데이터셋은 근무부담, 보상 수준, 조직 정착도, 직무 및 부서 특성 등 실무적으로 설명 가능한 변수 중심으로 재구성
- 이 단계의 목적 : 예측력뿐 아니라 해석 가능성과 활용 가능성을 함께 고려한 AutoML 적용용 변수셋 정리
- 이후 단계에서는 v2 데이터셋 기준으로 AutoML 적용 후 모델 성능 및 활용 가능성 비교 예정